# Food Portion Estimator — GUI App
## Pipeline: CNN (Keras) → YOLO → SAM-b → S/M/L + Gradio Interface

### Lỗi đã sửa so với notebook trước:
- **Lỗi 1**: Dùng `load_model()` thay vì rebuild + `load_weights` (training lưu full model kèm layer `data_augmentation`)
- **Lỗi 2**: Bỏ `/255.0` — EfficientNetB0 của Keras nhận input [0,255], không phải [0,1]

In [89]:
!pip install ultralytics gradio --quiet

In [90]:
import os, cv2, random, warnings
from ultralytics import YOLOWorld
import numpy as np
import pandas as pd
import torch
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')  # non-interactive backend cho Gradio
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from ultralytics import YOLO, SAM
import gradio as gr
warnings.filterwarnings('ignore')

print('TensorFlow :', tf.__version__)
print('Gradio     :', gr.__version__)

# ── Paths ──────────────────────────────────────────────────────
CNN_CKPT   = '/kaggle/input/models/minhquang2701/model2/keras/default/1/best_model.keras'
OUTPUT_DIR = Path('/kaggle/working/gui_output')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Classes — đúng thứ tự SELECTED_CLASSES lúc train ──────────
CLASSES = [
    'pizza', 'hamburger', 'french_fries', 'ice_cream', 'chocolate_cake',
    'sushi', 'ramen', 'fried_rice', 'omelette', 'pancakes',
    'hot_dog', 'grilled_salmon', 'caesar_salad', 'donuts', 'dumplings',
]
# ⚠ Thứ tự này khớp với SELECTED_CLASSES trong notebook train

YOLO_FOOD_IDS = {46,47,48,49,50,51,52,53,54,55}
THRESHOLDS    = {'small': 0.28, 'large': 0.55}
PORTION_COLOR = {'Small':'#3498db','Medium':'#f39c12','Large':'#e74c3c'}
PORTION_EMOJI = {'Small':'🔵 Small','Medium':'🟡 Medium','Large':'🔴 Large'}

WEIGHTS = {
    'R1_area_ratio'  : 0.50,
    'R2_bbox_fill'   : 0.25,
    'R3_compactness' : 0.05,
    'R4_saturation'  : 0.10,
    'R5_edge_density': 0.10,
}
CLASS_OFFSET = {
    'ramen': 0.08, 'pizza':0.06, 'fried_rice':0.05, 'pancakes':0.04,
    'caesar_salad':-0.05, 'sushi':-0.03, 'dumplings':-0.03,
}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch device:', DEVICE)

TensorFlow : 2.19.0
Gradio     : 5.50.0
PyTorch device: cuda


In [91]:
# ═══════════════════════════════════════════════════════
# LOAD CNN — fix Keras version mismatch
#
# Lỗi: load_model fail vì Dense config có 'quantization_config'
# mà Keras hiện tại không nhận → dùng load_weights thay thế
#
# Bắt buộc rebuild ĐÚNG kiến trúc training:
#   - mixed_float16 policy (training dùng mixed_precision)
#   - data_augmentation nằm TRONG model graph
#   - Dense output dtype=float32 (explicit như training)
# ═══════════════════════════════════════════════════════
from tensorflow.keras import mixed_precision, layers

# 1. Set đúng policy như lúc train
mixed_precision.set_global_policy('mixed_float16')
print('Policy:', mixed_precision.global_policy())

# 2. Rebuild data_augmentation (inference mode = no-op, không ảnh hưởng kết quả)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

# 3. Rebuild kiến trúc ĐÚNG như training
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights=None,               # không load ImageNet weights
    input_shape=(224, 224, 3)
)
base_model.trainable = False

inputs  = layers.Input(shape=(224, 224, 3))
x       = data_augmentation(inputs)
x       = base_model(x, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.2)(x)

NUM_CLASSES = len(CLASSES)
outputs = layers.Dense(
    NUM_CLASSES,
    activation='softmax',
    dtype=tf.float32            # explicit float32 như training
)(x)

cnn_model = tf.keras.Model(inputs, outputs)

# 4. Build trước khi load weights (tạo weight tensors)
cnn_model(tf.zeros((1, 224, 224, 3)), training=False)

# 5. Load weights từ .keras file — bỏ qua config, chỉ lấy weights
cnn_model.load_weights(CNN_CKPT)
cnn_model.trainable = False

print('✓ CNN loaded via load_weights')
print('  Input :', cnn_model.input_shape)
print('  Output:', cnn_model.output_shape)

# ── [B] YOLOv8n ──────────────────────────────────────
yolo_model = YOLO('yolov8n.pt')
print('✓ YOLOv8n loaded')

# ── [C] SAM-b ─────────────────────────────────────────
sam_model = SAM('sam_b.pt')
sam_model.to(DEVICE)
print('✓ SAM-b loaded on', DEVICE)

Policy: <DTypePolicy "mixed_float16">
✓ CNN loaded via load_weights
  Input : (None, 224, 224, 3)
  Output: (None, 15)
✓ YOLOv8n loaded
✓ SAM-b loaded on cuda


In [92]:
# ═══════════════════════════════════════════════════════
# PIPELINE FUNCTIONS
# ═══════════════════════════════════════════════════════

def preprocess_image(img_bgr, size=256):
    return cv2.resize(img_bgr, (size, size), interpolation=cv2.INTER_LINEAR)

# def preprocess_for_cnn(img_bgr):
#     """
#     Resize về 224×224 cho CNN.
#     ✅ FIX: KHÔNG chia /255 — EfficientNetB0 nhận [0,255]
#     khớp với training: tf.cast(image, tf.float32) không normalize.
#     """
#     img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
#     img_224 = cv2.resize(img_rgb, (224,224), interpolation=cv2.INTER_LINEAR)
#     return np.expand_dims(img_224.astype(np.float32), axis=0)  # (1,224,224,3)

def preprocess_for_cnn(img_bgr):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_224 = cv2.resize(img_rgb, (224, 224), interpolation=cv2.INTER_LINEAR)
    # float32, range [0,255] — EfficientNetB0 tự normalize nội bộ
    return np.expand_dims(img_224.astype(np.float32), axis=0)


def cnn_classify(img_bgr):
    x     = preprocess_for_cnn(img_bgr)
    preds = cnn_model.predict(x, verbose=0)[0]   # (15,)
    idx   = int(np.argmax(preds))
    return CLASSES[idx], float(preds[idx])


def yolo_detect(img_bgr, conf_thr=0.25):
    h, w    = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = yolo_model(img_rgb, conf=conf_thr, verbose=False)
    boxes   = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        xyxy    = boxes.xyxy.cpu().numpy()
        confs   = boxes.conf.cpu().numpy()
        cls_ids = boxes.cls.cpu().numpy().astype(int)
        food_mask = np.array([c in YOLO_FOOD_IDS for c in cls_ids])
        if food_mask.any():
            return xyxy[int(np.argmax(confs*food_mask))].tolist(), 'food_class'
        return xyxy[int(np.argmax(confs))].tolist(), 'any_object'
    m = 0.15
    return [int(w*m),int(h*m),int(w*(1-m)),int(h*(1-m))], 'center_fallback'


def expand_bbox(bbox, shape, margin=0.05):
    h, w = shape[:2]
    x1,y1,x2,y2 = bbox
    dx,dy = (x2-x1)*margin, (y2-y1)*margin
    return [max(0,int(x1-dx)), max(0,int(y1-dy)),
            min(w-1,int(x2+dx)), min(h-1,int(y2+dy))]


def sam_refine(img_bgr, bbox):
    h, w    = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    try:
        res   = sam_model(img_rgb, bboxes=[bbox], verbose=False)
        masks = res[0].masks
        if masks is not None and len(masks) > 0:
            mn = masks.data.cpu().numpy()
            b  = (mn[int(np.argmax([m.sum() for m in mn]))] * 255).astype(np.uint8)
        else:
            b = _bbox_mask(bbox, h, w)
    except:
        b = _bbox_mask(bbox, h, w)
    if b.shape != (h,w):
        b = cv2.resize(b,(w,h),interpolation=cv2.INTER_NEAREST)
    return b


def _bbox_mask(bbox, h, w):
    m = np.zeros((h,w), np.uint8)
    x1,y1,x2,y2 = [int(v) for v in bbox]
    m[y1:y2,x1:x2] = 255
    return m


def mask_cleanup(mask):
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15)))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n > 1:
        lg   = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        mask = np.where(labels==lg, np.uint8(255), np.uint8(0))
    return mask


def extract_ratios(img_bgr, mask, food_class=''):
    h, w  = img_bgr.shape[:2]
    total = h * w
    fg    = int(mask.sum() // 255)
    f     = {'food_class':food_class,'total_pixels':total,'fg_pixels':fg}
    f['R1_area_ratio'] = round(fg/total if total>0 else 0., 4)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        lg = max(cnts, key=cv2.contourArea)
        x,y,bw,bh = cv2.boundingRect(lg)
        f['R2_bbox_fill']   = round(min(fg/(bw*bh),1.) if bw*bh>0 else 0., 4)
        p = cv2.arcLength(lg, True)
        f['R3_compactness'] = round(min(4*np.pi*cv2.contourArea(lg)/p**2,1.)
                                    if p>0 else 0., 4)
    else:
        f['R2_bbox_fill'] = f['R3_compactness'] = 0.
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    fb  = mask > 0
    f['R4_saturation']  = round(float(hsv[:,:,1][fb].mean())/255. if fb.any() else 0., 4)
    edges = cv2.Canny(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), 50, 150)
    f['R5_edge_density']= round(
        float(cv2.bitwise_and(edges,edges,mask=mask).sum()//255)/fg
        if fg>0 else 0., 4)
    return f


def score_and_classify(feats, food_class=''):
    score  = round(sum(WEIGHTS[k]*feats.get(k,0.) for k in WEIGHTS), 4)
    off    = CLASS_OFFSET.get(food_class, 0.)
    if score < THRESHOLDS['small'] + off: portion = 'Small'
    elif score > THRESHOLDS['large'] + off: portion = 'Large'
    else: portion = 'Medium'
    return score, portion


# def run_pipeline(img_bgr):
#     """Full pipeline, trả về dict kết quả."""
#     pre              = preprocess_image(img_bgr, 512)
#     food_class, conf = cnn_classify(img_bgr)
#     raw_bbox, src    = yolo_detect(pre)
#     bbox             = expand_bbox(raw_bbox, pre.shape)
#     mask_raw         = sam_refine(pre, bbox)
#     mask             = mask_cleanup(mask_raw)
#     feats            = extract_ratios(pre, mask, food_class)
#     score, portion   = score_and_classify(feats, food_class)
#     return dict(
#         pre=pre, mask=mask, mask_raw=mask_raw,
#         food_class=food_class, cnn_conf=conf,
#         bbox=bbox, yolo_src=src,
#         score=score, portion=portion, **feats
#     )

def run_pipeline(img_bgr):
    pre              = preprocess_image(img_bgr, 512)
    food_class, conf = cnn_classify(img_bgr)
    raw_bbox, src    = yolo_detect(pre)
    bbox             = expand_bbox(raw_bbox, pre.shape)
    mask_raw         = sam_refine(pre, bbox)
    mask             = mask_cleanup(mask_raw)
    feats            = extract_ratios(pre, mask, food_class)
    score, portion   = score_and_classify(feats, food_class)

    return {
        # loại food_class khỏi feats trước khi unpack để tránh trùng key
        **{k: v for k, v in feats.items() if k != 'food_class'},
        'food_class' : food_class,
        'cnn_conf'   : conf,
        'bbox'       : bbox,
        'yolo_src'   : src,
        'score'      : score,
        'portion'    : portion,
        'pre'        : pre,
        'mask'       : mask,
        'mask_raw'   : mask_raw,
    }

In [93]:
# ═══════════════════════════════════════════════════════
# VISUALIZATION HELPERS — trả về PIL.Image cho Gradio
# ═══════════════════════════════════════════════════════

def make_detection_image(r):
    """Ảnh gốc + YOLO bbox (xanh lá) + CNN label."""
    vis = r['pre'].copy()
    x1,y1,x2,y2 = [int(v) for v in r['bbox']]
    cv2.rectangle(vis, (x1,y1),(x2,y2), (34,197,94), 3)
    label = f"{r['food_class']} {r['cnn_conf']:.0%}"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(vis, (x1, y1-th-10),(x1+tw+6, y1), (34,197,94), -1)
    cv2.putText(vis, label, (x1+3, y1-5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)
    return Image.fromarray(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))


def make_mask_overlay(r):
    """Ảnh + SAM mask overlay màu xanh lam bán trong suốt."""
    img_rgb  = cv2.cvtColor(r['pre'], cv2.COLOR_BGR2RGB)
    overlay  = img_rgb.copy()
    color    = np.array([52, 152, 219], dtype=np.uint8)  # #3498db
    fg       = r['mask'] > 0
    overlay[fg] = (img_rgb[fg] * 0.45 + color * 0.55).astype(np.uint8)
    # Vẽ contour trắng
    cnts, _  = cv2.findContours(r['mask'], cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, cnts, -1, (255,255,255), 2)
    return Image.fromarray(overlay)


# def make_ratio_chart(r):
#     """Horizontal bar chart 5 ratios — trả về PIL.Image."""
#     names  = ['R1  Area ratio','R2  BBox fill',
#               'R3  Compactness','R4  Saturation','R5  Edge density']
#     values = [r['R1_area_ratio'], r['R2_bbox_fill'],
#               r['R3_compactness'], r['R4_saturation'],
#               r['R5_edge_density']]
#     color  = PORTION_COLOR[r['portion']]

#     fig, ax = plt.subplots(figsize=(5, 3.2))
#     bars = ax.barh(names, values, color=color, alpha=0.85, height=0.6)
#     ax.set_xlim(0, 1)
#     ax.axvline(THRESHOLDS['small'], color='#95a5a6',
#                linestyle='--', linewidth=1.2, label=f'T_small={THRESHOLDS["small"]}')
#     ax.axvline(THRESHOLDS['large'], color='#2c3e50',
#                linestyle='--', linewidth=1.2, label=f'T_large={THRESHOLDS["large"]}')
#     # Ghi số lên mỗi bar
#     for bar, val in zip(bars, values):
#         ax.text(min(val+0.02, 0.92), bar.get_y()+bar.get_height()/2,
#                 f'{val:.3f}', va='center', fontsize=9, color='#2c3e50')
#     ax.set_xlabel('Ratio value', fontsize=9)
#     ax.set_title(f'Weighted Score: {r["score"]:.3f}', fontsize=10, fontweight='bold')
#     ax.legend(fontsize=8, loc='lower right')
#     ax.spines[['top','right']].set_visible(False)
#     plt.tight_layout()

#     fig.canvas.draw()
#     buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
#     buf = buf.reshape(fig.canvas.get_width_height()[::-1] + (3,))
#     plt.close(fig)
#     return Image.fromarray(buf)

def make_ratio_chart(r):
    import io
    names  = ['R1  Area ratio','R2  BBox fill',
              'R3  Compactness','R4  Saturation','R5  Edge density']
    values = [r['R1_area_ratio'], r['R2_bbox_fill'],
              r['R3_compactness'], r['R4_saturation'],
              r['R5_edge_density']]
    color  = PORTION_COLOR[r['portion']]

    fig, ax = plt.subplots(figsize=(5, 3.2))
    bars = ax.barh(names, values, color=color, alpha=0.85, height=0.6)
    ax.set_xlim(0, 1)
    ax.axvline(THRESHOLDS['small'], color='#95a5a6',
               linestyle='--', linewidth=1.2,
               label=f'T_small={THRESHOLDS["small"]}')
    ax.axvline(THRESHOLDS['large'], color='#2c3e50',
               linestyle='--', linewidth=1.2,
               label=f'T_large={THRESHOLDS["large"]}')
    for bar, val in zip(bars, values):
        ax.text(min(val+0.02, 0.92), bar.get_y()+bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9, color='#2c3e50')
    ax.set_xlabel('Ratio value', fontsize=9)
    ax.set_title(f'Weighted Score: {r["score"]:.3f}',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()

    # ── Dùng BytesIO thay tostring_rgb (đã bị xóa ở matplotlib mới) ──
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf).copy()   # .copy() để tránh lỗi khi buf bị close
    plt.close(fig)
    return img

def make_result_summary(r):
    """Text markdown summary."""
    p      = r['portion']
    emoji  = PORTION_EMOJI[p]
    clr    = PORTION_COLOR[p]
    return (
        f"## {emoji}\n\n"
        f"| Thông tin | Giá trị |\n"
        f"|---|---|\n"
        f"| 🍽 Món ăn | **{r['food_class']}** |\n"
        f"| 🎯 CNN confidence | **{r['cnn_conf']:.1%}** |\n"
        f"| 📦 YOLO source | {r['yolo_src']} |\n"
        f"| 📊 Weighted score | **{r['score']:.3f}** |\n"
        f"| 📏 Area ratio (R1) | {r['R1_area_ratio']:.2%} |\n"
        f"| 🔲 BBox fill (R2) | {r['R2_bbox_fill']:.2%} |\n"
    )

In [94]:
# ═══════════════════════════════════════════════════════
# GRADIO GUI — Food Portion Estimator
# ═══════════════════════════════════════════════════════

def analyze(pil_image):
    """
    Hàm xử lý chính — Gradio gọi mỗi khi nhấn Analyze.
    Input : PIL.Image (từ Gradio upload)
    Output: 4 outputs tương ứng 4 component trong interface
    """
    if pil_image is None:
        return None, None, None, '## ⚠ Vui lòng upload ảnh'

    # PIL → BGR numpy
    img_bgr = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)

    try:
        r = run_pipeline(img_bgr)
    except Exception as e:
        return None, None, None, f'## ❌ Lỗi: {e}'

    det_img   = make_detection_image(r)
    mask_img  = make_mask_overlay(r)
    chart_img = make_ratio_chart(r)
    summary   = make_result_summary(r)

    return det_img, mask_img, chart_img, summary


# ── CSS tùy chỉnh giao diện ──────────────────────────────────────
CUSTOM_CSS = """
#title { text-align: center; }
#title h1 { color: #2c3e50; font-size: 1.8em; margin-bottom: 4px; }
#title p  { color: #7f8c8d; font-size: 0.95em; }
#upload-col { background: #f8f9fa; border-radius: 12px; padding: 16px; }
#result-col { background: #ffffff; border-radius: 12px; padding: 16px; }
#analyze-btn { background: #2ecc71 !important; color: white !important;
               font-size: 1.1em !important; border-radius: 8px !important;
               height: 48px !important; }
#analyze-btn:hover { background: #27ae60 !important; }
#portion-box { border-radius: 10px; padding: 16px;
               background: #f0f8ff; border: 1px solid #d6eaf8; }

/* ── Toàn bộ text trong cột kết quả ── */
#result-col, #result-col * { color: #1a1a1a !important; }

/* ── Markdown summary (portion label) ── */
#portion-box, #portion-box * { color: #1a1a1a !important; }

/* ── Label của các image component ── */
.output-image label span { color: #1a1a1a !important; }

/* ── Text trong gr.Markdown toàn trang ── */
.prose, .prose * { color: #1a1a1a !important; }

/* ── Table trong summary ── */
table, th, td { color: #1a1a1a !important; }
"""

# ── Build interface với gr.Blocks ────────────────────────────────
with gr.Blocks(css=CUSTOM_CSS, title='Food Portion Estimator') as demo:

    # ── Header ──────────────────────────────────────────────────
    with gr.Column(elem_id='title'):
        gr.HTML("""
        <h1>🍽 Food Portion Estimator</h1>
        <p>Upload ảnh món ăn → CNN nhận diện → YOLO detect → SAM-b segment → Ước lượng khẩu phần</p>
        <p style='font-size:0.8em; color:#95a5a6;'>
            Hỗ trợ 15 class: pizza · hamburger · french_fries · ice_cream · chocolate_cake ·
            sushi · ramen · fried_rice · omelette · pancakes ·
            hot_dog · grilled_salmon · caesar_salad · donuts · dumplings
        </p>
        """)

    gr.Markdown('---')

    # ── Main layout: 2 cột ──────────────────────────────────────
    with gr.Row():

        # ── Cột trái: Input ─────────────────────────────────────
        with gr.Column(scale=1, elem_id='upload-col'):
            gr.Markdown('### 📤 Input')
            img_input = gr.Image(
                type='pil',
                label='Upload ảnh món ăn',
                height=300,
                sources=['upload', 'webcam', 'clipboard'],
            )
            analyze_btn = gr.Button(
                '🔍  Analyze',
                variant='primary',
                elem_id='analyze-btn',
            )
            gr.Markdown("""
            **Hướng dẫn:**
            1. Upload ảnh hoặc chụp từ webcam
            2. Nhấn **Analyze**
            3. Chờ ~5-10 giây để pipeline xử lý
            """)

            # ── Examples ────────────────────────────────────────
            gr.Examples(
                examples=[
                    '/kaggle/input/datasets/minhquang2701/food101-15cla/pizza/'
                    + str(sorted(Path('/kaggle/input/datasets/minhquang2701/'
                                     'food101-15cla/pizza').glob('*.jpg'))[0].name)
                    if Path('/kaggle/input/datasets/minhquang2701/'
                            'food101-15cla/pizza').exists() else None,
                ],
                inputs=img_input,
                label='Ảnh mẫu',
            )

        # ── Cột phải: Output ─────────────────────────────────────
        with gr.Column(scale=2, elem_id='result-col'):
            gr.Markdown('### 📊 Kết quả phân tích')

            # Dòng 1: Summary
            with gr.Row():
                portion_md = gr.Markdown(
                    '### Chưa có kết quả — upload ảnh và nhấn Analyze',
                    elem_id='portion-box',
                )

            # Dòng 2: 2 ảnh detection + mask
            with gr.Row():
                det_out = gr.Image(
                    label='[1] CNN + [2] YOLO Detection',
                    height=240,
                    show_download_button=True,
                )
                mask_out = gr.Image(
                    label='[3] SAM-b Mask Overlay',
                    height=240,
                    show_download_button=True,
                )

            # Dòng 3: Ratio chart full width
            with gr.Row():
                chart_out = gr.Image(
                    label='[5] Geometry Ratios → [6] Portion Score',
                    height=220,
                    show_download_button=True,
                )

    # ── Event binding ────────────────────────────────────────────
    analyze_btn.click(
        fn=analyze,
        inputs=[img_input],
        outputs=[det_out, mask_out, chart_out, portion_md],
    )
    # Cũng analyze khi upload xong (tùy chọn — comment nếu muốn manual)
    # img_input.upload(
    #     fn=analyze,
    #     inputs=[img_input],
    #     outputs=[det_out, mask_out, chart_out, portion_md],
    # )

    gr.Markdown("""
    ---
    <small>Pipeline: EfficientNet-B0 (Keras) → YOLOv8n → SAM-b → 5 Geometry Ratios → Class-aware Scoring</small>
    """)

print('✓ Gradio app built')

✓ Gradio app built


In [95]:
# ═══════════════════════════════════════════════════════
# LAUNCH APP
# share=True tạo link public để share demo
# ═══════════════════════════════════════════════════════
demo.launch(
    share=True,            # tạo link gradio.live để demo
    show_error=True,       # hiển thị lỗi chi tiết khi debug
    quiet=False,
)

* Running on local URL:  http://127.0.0.1:7870
* Running on public URL: https://147fb50148287b4e87.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [96]:
# ═══════════════════════════════════════════════════════
# (TÙY CHỌN) Test pipeline + lưu ảnh kết quả ra file
# Chạy cell này để kiểm tra trước khi launch GUI
# ═══════════════════════════════════════════════════════
test_cls  = 'pizza'
test_dir  = Path('/kaggle/input/datasets/minhquang2701/food101-15cla') / test_cls
test_path = sorted(test_dir.glob('*.jpg'))[0]

test_bgr = cv2.imread(str(test_path))
r        = run_pipeline(test_bgr)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

det = np.array(make_detection_image(r))
axes[0].imshow(det)
axes[0].set_title(f'CNN: {r["food_class"]} ({r["cnn_conf"]:.1%})\n'
                  f'YOLO: {r["yolo_src"]}')
axes[0].axis('off')

msk = np.array(make_mask_overlay(r))
axes[1].imshow(msk)
axes[1].set_title(f'SAM mask  area={r["R1_area_ratio"]:.2%}')
axes[1].axis('off')

chart = np.array(make_ratio_chart(r))
axes[2].imshow(chart)
axes[2].set_title(f'Portion: {r["portion"]}  score={r["score"]:.3f}')
axes[2].axis('off')

plt.suptitle(f'Pipeline Test — {test_cls}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pipeline_test.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✓ Saved → {OUTPUT_DIR}/pipeline_test.png')

IndexError: list index out of range

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error